# H0 dense TabPFN screen

## tl;dr
H0의 규정 안전 EB/구조 증거를 저차원 dense 입력으로 고정 TabPFN에 전달해 오류 다양성을 확인합니다. 이 노트북은 seed42 screen만 실행하며 test.csv를 읽지 않습니다.

## Context & Methods

### Key Assumptions
- H0는 seed42 OOF 0.547915 ± 0.001을 먼저 재현해야 합니다.
- vocabulary, EB, scaling은 outer-fold train only입니다.
- fixed `0.80 H0 + 0.20 TabPFN`만 사용하며 HPO·AutoTabPFN·가중치 탐색은 하지 않습니다.
- 특정 class/gene/exact mutation 규칙과 test 통계는 사용하지 않습니다.

In [1]:
from pathlib import Path
import importlib.util, json, subprocess, sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

ROOT = Path('/Users/admin/Documents/FinalProject/OZ_fianl_hackaton')
RUNNER = ROOT / 'experiments/gs/notebooks/exp_model_013/common/run_h0_dense_tabpfn_screen.py'
RESULT = ROOT / 'experiments/gs/notebooks/exp_model_013/result'
RUN_ID = 'exp-h0-dense-tabpfn-screen-01'
RUN_EXPERIMENT = True  # 준비를 확인한 뒤 True로 변경
DEVICE = 'cuda'  # CUDA가 없으면 'cpu'; CPU 전체 CV는 매우 느릴 수 있습니다.
assert RUNNER.exists()
print({'runner': RUNNER, 'result': RESULT, 'tabpfn_installed': importlib.util.find_spec('tabpfn') is not None})

{'runner': PosixPath('/Users/admin/Documents/FinalProject/OZ_fianl_hackaton/experiments/gs/notebooks/exp_model_013/common/run_h0_dense_tabpfn_screen.py'), 'result': PosixPath('/Users/admin/Documents/FinalProject/OZ_fianl_hackaton/experiments/gs/notebooks/exp_model_013/result'), 'tabpfn_installed': True}


### 1. Dependency check

`tabpfn`이 없으면 먼저 아래 주석 명령을 별도 셀/터미널에서 실행하세요. 첫 fit에서 공식 checkpoint 다운로드 및 라이선스 인증이 필요할 수 있습니다.

`/Users/admin/Documents/FinalProject/OZ_fianl_hackaton/.venv/bin/python -m pip install tabpfn`

In [2]:
smoke = subprocess.run([sys.executable, str(RUNNER), '--smoke'], text=True, capture_output=True, check=True)
print(smoke.stdout)
assert 'test_read' in smoke.stdout and 'nan_as_mutation_count' in smoke.stdout

{"test_read": false, "train_test_concat": false, "fixed_class_gene_mutation_rules": false, "vocabulary_source": "outer_fold_train_only", "supervised_eb_source": "outer_fold_train_only_with_inner_crossfit", "scaler_source": "outer_fold_train_only", "outer_validation_used_for_fit": false, "leakage_check": true, "nan_as_mutation_count": 0, "tabpfn_weight": 0.2, "h0_weight": 0.8, "tabpfn_model_contract": "fixed_tabpfn_v3_classifier_no_hpo_no_autotabpfn", "example_dense_feature_count": 2}



### 2. Run seed42 screen

H0 재현이 기준에서 벗어나면 TabPFN fit을 시작하지 않습니다. fold별로 진행 상황을 출력하며, H0-only checkpoint를 남겨 dependency 오류 뒤에도 불필요한 재학습을 피합니다.

In [3]:
if RUN_EXPERIMENT:
    command = [sys.executable, str(RUNNER), '--run-id', RUN_ID, '--device', DEVICE]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in process.stdout:
        print(line, end='')
        tail = (tail + [line])[-120:]
    if process.wait():
        raise RuntimeError('TabPFN screen failed:\n' + ''.join(tail))
else:
    print('RUN_EXPERIMENT=False: result files are read only.')

[TabPFN screen] H0 reproduction fold 1/5
[TabPFN screen] H0 reproduction fold 2/5
[TabPFN screen] H0 reproduction fold 3/5
[TabPFN screen] H0 reproduction fold 4/5
[TabPFN screen] H0 reproduction fold 5/5
[TabPFN screen] fold 1/5: fold-train dense evidence + TabPFN
Traceback (most recent call last):
  File "/Users/admin/Documents/FinalProject/OZ_fianl_hackaton/.venv/lib/python3.12/site-packages/tabpfn/browser_auth.py", line 164, in _get_license_name
    with urllib.request.urlopen(req, timeout=10) as resp:  # noqa: S310
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/admin/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/urllib/request.py", line 215, in urlopen
    return opener.open(url, data, timeout)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/admin/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/urllib/request.py", line 521, in open
    response = meth(req, response)
               ^^^^^^^^^^^^^^^^^^^
  

RuntimeError: TabPFN screen failed:
[TabPFN screen] H0 reproduction fold 1/5
[TabPFN screen] H0 reproduction fold 2/5
[TabPFN screen] H0 reproduction fold 3/5
[TabPFN screen] H0 reproduction fold 4/5
[TabPFN screen] H0 reproduction fold 5/5
[TabPFN screen] fold 1/5: fold-train dense evidence + TabPFN
Traceback (most recent call last):
  File "/Users/admin/Documents/FinalProject/OZ_fianl_hackaton/.venv/lib/python3.12/site-packages/tabpfn/browser_auth.py", line 164, in _get_license_name
    with urllib.request.urlopen(req, timeout=10) as resp:  # noqa: S310
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/admin/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/urllib/request.py", line 215, in urlopen
    return opener.open(url, data, timeout)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/admin/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/urllib/request.py", line 521, in open
    response = meth(req, response)
 

## Results

In [ ]:
prefix = RESULT / f'{RUN_ID}_seed42'
summary_path = prefix.with_name(prefix.name + '_summary.csv')
if summary_path.exists():
    summary = pd.read_csv(summary_path)
    folds = pd.read_csv(prefix.with_name(prefix.name + '_fold_metrics.csv'))
    topk = pd.read_csv(prefix.with_name(prefix.name + '_topk_metrics.csv'))
    decision = json.loads(prefix.with_name(prefix.name + '_leakage_audit.json').read_text())
    assert summary.leakage_check.all() and summary.nan_as_mutation_count.eq(0).all()
    display(summary)
    display(topk)
    display(pd.DataFrame([decision]))
else:
    print('No result yet. Set RUN_EXPERIMENT=True after TabPFN installation and checkpoint access are ready.')

In [ ]:
for suffix in ('_fold_macro_f1.png', '_class_f1_delta.png', '_topk_recall.png'):
    image = prefix.with_name(prefix.name + suffix)
    if image.exists():
        display(Image(filename=str(image)))

## Takeaways

- `screen_candidate`는 H0 대비 +0.015 이상, 4/5 fold 양수일 때만 부여됩니다.
- 미통과이면 TabPFN의 재튜닝이나 blend 비율 탐색은 하지 않고 축을 종료합니다.
- 통과하면 동일 설정 그대로 42/777/2024 3-seed 검증으로만 확장합니다.